# Walk-forward Trend, Ridge and minute PPO training

Upload the extended portfolio `.tar` and walk-forward inputs `.zip` to `MyDrive/Stockmog`.
Select **Runtime > Change runtime type > GPU**, then run the cells in order.
No real trades are placed. Outputs and checkpoints are saved to a new Drive folder.

The run trains 36 Trend Analyzer jobs in three chronological folds (108 hybrid fits),
then Ridge and one shared-cash minute PPO using **50 vectorized environments**.
This can require many hours, substantial RAM and multiple Colab sessions. Colab may disconnect;
rerun the notebook with unchanged settings to resume checkpoints. The 50 environments share
one PPO policy; they are not 50 independently trained models or 50 CPU processes.

The first 20 validation sessions were already inspected. The test period starting
2025-08-11 is excluded from fitting and evaluation, but prior research limits its independence.
News and research are incomplete, with no matched M&M news events in these inputs;
adjusted-price trading simulations are approximate. Daily history starts in August 2020,
common minute history in January 2022, and out-of-fold trend forecasts in January 2023.
Earlier forecasts are explicitly masked, not generated using later-trained models.
PPO marks open positions to market and does not apply Ridge's per-stock cap, so their
displayed returns are not directly comparable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/Stockmog')
ARCHIVE = DRIVE / 'ppo-portfolio-top4-extended-20261124.tar'
ADDON_ARCHIVE = DRIVE / 'walkforward-training-inputs-20261124.zip'
OUTPUT = DRIVE / 'walkforward-training-20261124-v1'
WORK = Path('/content/stockmog-walkforward-v1')
ARCHIVE_SHA = '395e7f80ff956d9b1ecb5ba422d9b556c0d91e74e42963b14e4e85da6a20bb57'
ADDON_SHA = 'f8d3d2cf144d02d334c87307c8f463b2678182840d2cf086683ec645cf488c55'
TREND_EPOCHS = 20
TREE_ROUNDS = 100
PPO_STEPS = 256_000
RUN_TREND = True
RUN_RIDGE = True
RUN_PPO = True

In [ ]:
import hashlib, json, shutil, tarfile, zipfile

def file_hash(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

for path, expected in ((ARCHIVE, ARCHIVE_SHA), (ADDON_ARCHIVE, ADDON_SHA)):
    if not path.is_file() or file_hash(path) != expected:
        raise ValueError(f'Missing or corrupt uploaded archive: {path}')
WORK.mkdir(parents=True, exist_ok=True)
if shutil.disk_usage(WORK).free < 3 * 1024**3:
    raise RuntimeError('At least 3 GiB of free Colab disk is required')
BUNDLE = WORK / 'stockmog-ppo'
SUPPLEMENT = WORK / 'supplement'
marker = WORK / 'portfolio-archive-sha256.txt'
if BUNDLE.exists():
    if not marker.is_file() or marker.read_text().strip() != ARCHIVE_SHA:
        raise RuntimeError('Portfolio extraction incomplete or different; use a fresh WORK path')
else:
    with tarfile.open(ARCHIVE) as archive:
        for member in archive.getmembers():
            if not (member.isfile() or member.isdir()) or not (
                    member.name == 'stockmog-ppo' or member.name.startswith('stockmog-ppo/')):
                raise ValueError(f'Unexpected tar entry: {member.name}')
        archive.extractall(WORK, filter='data')
    marker.write_text(ARCHIVE_SHA)
marker = WORK / 'supplement-archive-sha256.txt'
if SUPPLEMENT.exists():
    if not marker.is_file() or marker.read_text().strip() != ADDON_SHA:
        raise RuntimeError('Supplement extraction incomplete or different; use a fresh WORK path')
else:
    SUPPLEMENT.mkdir()
    with zipfile.ZipFile(ADDON_ARCHIVE) as archive:
        for member in archive.infolist():
            target = (SUPPLEMENT / member.filename).resolve()
            if not target.is_relative_to(SUPPLEMENT.resolve()) or member.is_dir():
                raise ValueError(f'Unexpected ZIP entry: {member.filename}')
        archive.extractall(SUPPLEMENT)
    marker.write_text(ADDON_SHA)
payload = json.loads((SUPPLEMENT / 'payload-manifest.json').read_text())
for name, expected in payload['files'].items():
    if file_hash(SUPPLEMENT / name) != expected:
        raise ValueError(f'Supplement checksum mismatch: {name}')
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
if manifest['status'] != 'complete' or len(manifest['jobs']) != 36:
    raise ValueError('Unexpected portfolio archive')
for name, expected in manifest['files'].items():
    if name.startswith('source/') and file_hash(BUNDLE / name) != expected:
        raise ValueError(f'Archived source checksum mismatch: {name}')
print('Input archives verified. News/research coverage is partial.')

In [ ]:
import importlib.metadata, os, subprocess, sys
from packaging.requirements import Requirement

SOURCE = BUNDLE / 'source'
def unmet_requirements(path):
    missing = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('-r '):
            missing.extend(unmet_requirements(path.parent / line[3:].strip()))
            continue
        requirement = Requirement(line)
        if requirement.marker and not requirement.marker.evaluate():
            continue
        try:
            version = importlib.metadata.version(requirement.name)
        except importlib.metadata.PackageNotFoundError:
            missing.append(line)
            continue
        if requirement.specifier and not requirement.specifier.contains(version):
            missing.append(line)
    return missing

missing = unmet_requirements(SOURCE / 'requirements-rl.txt')
if missing:
    print('Installing missing/incompatible dependencies:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                           str(SOURCE / 'requirements-rl.txt')])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU unavailable. Enable a GPU runtime and rerun setup.')
print('GPU:', torch.cuda.get_device_name(0))
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['PYTHONPATH'] = os.pathsep.join([str(SOURCE), str(SUPPLEMENT)])
OUTPUT.mkdir(parents=True, exist_ok=True)

## Train and validate
The next cells save progress to Drive. Do not change epochs, rounds, PPO steps, data,
or output path when resuming an existing run. To experiment with different settings, use
a fresh output folder. Do not run two notebooks against the same output folder at once.
Trend jobs run sequentially; PPO uses 50 environments.

In [ ]:
if RUN_TREND:
    subprocess.check_call([sys.executable, str(SUPPLEMENT / 'walkforward_trend_training.py'),
                           '--bundle', str(BUNDLE), '--earlier', str(SUPPLEMENT / 'earlier'),
                           '--output', str(OUTPUT / 'trends'), '--device', 'cuda',
                           '--epochs', str(TREND_EPOCHS), '--tree-rounds', str(TREE_ROUNDS)])

In [ ]:
if RUN_RIDGE:
    subprocess.check_call([sys.executable, str(SUPPLEMENT / 'walkforward_consumers.py'),
                           '--stage', 'ridge', '--bundle', str(BUNDLE), '--supplement', str(SUPPLEMENT),
                           '--trends', str(OUTPUT / 'trends'), '--output', str(OUTPUT), '--device', 'cuda'])

In [ ]:
if RUN_PPO:
    subprocess.check_call([sys.executable, str(SUPPLEMENT / 'walkforward_consumers.py'),
                           '--stage', 'ppo', '--bundle', str(BUNDLE), '--supplement', str(SUPPLEMENT),
                           '--trends', str(OUTPUT / 'trends'), '--output', str(OUTPUT),
                           '--device', 'cuda', '--steps', str(PPO_STEPS)])

In [ ]:
import json
import pandas as pd
from IPython.display import display
ridge = OUTPUT / 'ridge' / 'validation-comparison.csv'
ppo = OUTPUT / 'ppo' / 'validation-summary.json'
if ridge.exists():
    print('Ridge validation (daily decisions):')
    display(pd.read_csv(ridge))
if ppo.exists():
    print('PPO validation (minute decisions):')
    print(json.dumps(json.loads(ppo.read_text()), indent=2))
print('Test-period performance was not evaluated.')